# Phase 6.5 shard 14 (forest65)

Runs **108 cells** of the frozen Phase 6.5 manifest (`G3-PHASE65-v1`), covering: `causal_drf`, `causal_drf_log`, `causal_drf_retn`, `drf`, `drf_log`.

This shard runs the R forest baselines, including the two adversarial controls (log geometry and bandwidth retune). The setup cell installs R, the pinned `drf` 1.3.1, and the authors' causal-clean package at the frozen commit; fifteen to twenty-five minutes.

Estimated single-threaded compute on the reference machine is about **87 minutes**. Colab cores are slower, so allow two to three times that, plus any install time above. This fits comfortably inside a nine hour session.

**Run every cell in order.** The last cell downloads a `.zip`; collect every shard's zip into `results/phase65/colab_shards/` (logs into `results/manifests/`) and run `python research/run_phase65.py merge`.


In [ ]:
# Thread pinning MUST happen before NumPy or SciPy are imported.
# OpenMP sizes its pool at initialisation, so setting these
# afterwards is silently ineffective.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_v] = '1'
print('threads pinned to 1')

## 1. Clone the repository at the pinned commit

Remote `https://github.com/hugogobato/wasserstein-causal-forests.git`, commit `bfe99cb3f305`. After checkout the notebook asserts the frozen manifest checksum, so a clone of anything but the generating commit fails here rather than mid-run.

In [ ]:
import subprocess, pathlib, os, sys, json, hashlib

REPO = 'https://github.com/hugogobato/wasserstein-causal-forests.git'
COMMIT = 'bfe99cb3f305d5f9372dfb723888ed128b8f6ed9'
EXPECTED_CHECKSUM = '4e28d308ca99cde4c81379524fc4492a15b38f029b449899b0a307b6c0ace110'

workdir = pathlib.Path('/content/wcf')
if not workdir.exists():
    subprocess.run(['git', 'init', '-q', str(workdir)], check=True)
    subprocess.run(
        ['git', '-C', str(workdir), 'remote', 'add', 'origin', REPO],
        check=True,
    )
# A shallow fetch of the exact commit: nothing else is downloaded.
    subprocess.run(
        ['git', '-C', str(workdir), 'fetch', '-q', '--depth', '1',
         'origin', COMMIT], check=True,
    )
    subprocess.run(
        ['git', '-C', str(workdir), 'checkout', '-q', 'FETCH_HEAD'],
        check=True,
    )
os.chdir(workdir)
sys.path.insert(0, str(workdir / 'src'))
os.environ['WCF_CAUSAL_DRF_R_LIB'] = '/content/wcf/results/Rlib/causal_drf'

manifest = json.load(open(
    'results/manifests/phase65_manifest.json', encoding='utf-8'
))
checksum = hashlib.sha256(
    json.dumps(manifest['cells'], sort_keys=True).encode('utf-8')
).hexdigest()
assert checksum == EXPECTED_CHECKSUM, (
    'the cloned manifest does not match the frozen grid: '
    f'{checksum} != {EXPECTED_CHECKSUM}'
)
print('repo ready at commit ' + COMMIT[:12] + '; '
      + str(manifest['n_cells']) + ' frozen cells verified')

## 2. Dependencies

In [ ]:
# This group runs the R forest baselines, including Causal-DRF
# through the authors' causal-clean package at the frozen commit.
# The causal-clean repository is a monorepo whose R package sits in
# r-package/drf, so the installer fetches the exact-commit tarball
# from codeload (no GitHub API, hence no shared-IP rate limit) and
# runs R CMD INSTALL on that subdirectory. Expect fifteen to twenty-
# five minutes for this cell.
%%bash
set -e
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev curl > /dev/null 2>&1
Rscript -e 'options(Ncpus=2); install.packages(c("Rcpp","RcppEigen","jsonlite","remotes","transport","fastDummies","kernlab"), repos="https://cloud.r-project.org", quiet=TRUE)'
# CRAN drf 1.3.1 drives the paper-DRF and W-DRF-T drivers; the
# causal-clean library below shadows it only for Causal-DRF cells.
Rscript -e 'options(Ncpus=2); if (!requireNamespace("drf", quietly=TRUE)) install.packages("drf", repos="https://cloud.r-project.org", quiet=TRUE); cat("CRAN drf", as.character(packageVersion("drf")), "ready\n")'
mkdir -p results/Rlib/causal_drf
CAUSAL_SHA="0a1a508444176b5b1553f13e832be93a374b0af2"
if [ ! -d results/Rlib/causal_drf/drf ]; then
  TARBALL="/tmp/causal_clean_${CAUSAL_SHA:0:12}.tar.gz"
  curl -sL "https://codeload.github.com/herbps10/drf/tar.gz/${CAUSAL_SHA}" -o "$TARBALL"
  EXTRACT="/tmp/causal_clean_src"
  rm -rf "$EXTRACT"; mkdir -p "$EXTRACT"
  tar -xzf "$TARBALL" -C "$EXTRACT"
  PKG_DIR=$(find "$EXTRACT" -maxdepth 3 -type d -path "*r-package/drf" | head -1)
  echo "installing causal-clean drf from $PKG_DIR"
  R CMD INSTALL --library=results/Rlib/causal_drf "$PKG_DIR" \
    || Rscript -e 'options(Ncpus=2); .libPaths(c("results/Rlib/causal_drf",.libPaths())); remotes::install_github("herbps10/drf", ref="0a1a508444176b5b1553f13e832be93a374b0af2", subdir="r-package/drf", lib="results/Rlib/causal_drf", upgrade="never", quiet=TRUE)'
fi
Rscript -e '.libPaths(c("results/Rlib/causal_drf",.libPaths())); stopifnot(requireNamespace("drf", quietly=TRUE)); cat("causal-clean drf", as.character(packageVersion("drf")), "ready\n")'
echo 'setup complete'


## 3. This shard's cells

In [ ]:
import json, collections
SHARD_INDEX = 14
CELLS = json.loads('''[{"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "a27b6a7643ed6097", "test_seed": 900002}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "61666eab77145399", "test_seed": 900002}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "474dc036d2efc48b", "test_seed": 900002}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "37822712305519cf", "test_seed": 900002}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "7b81c668aab08aa9", "test_seed": 900002}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "beecd5bfbc0854ef", "test_seed": 900002}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "8e7b0122b7ea1e3c", "test_seed": 900002}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "1e42b1c0b6ea06a7", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 2, "cell_key": "93eea0e3a3871916", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 7, "cell_key": "54da7317c82d03e6", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 2, "cell_key": "a8b691482271ced6", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 7, "cell_key": "3dcf238f12e7b602", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 2, "cell_key": "14850304e654a26c", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 7, "cell_key": "bb8b5135cfea93f3", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 2, "cell_key": "2855314a7b49991d", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 7, "cell_key": "7751fd7f50203a38", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 2, "cell_key": "3d96b9cf413a3c4d", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 7, "cell_key": "03817539495ceaeb", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 2, "cell_key": "6daebf88febb3e92", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 7, "cell_key": "4e4d635d052d9b22", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 2, "cell_key": "b39521b2a9607be4", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 7, "cell_key": "ccb2971e2542a192", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 2, "cell_key": "5539d7a9cd0086e7", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 7, "cell_key": "eec7ec1fcdce948b", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 2, "cell_key": "262ac1e1fba31946", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 7, "cell_key": "51b5f0f5fd4dfa66", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 2, "cell_key": "50727bb245aa5b81", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 7, "cell_key": "18786d88905d7c10", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 2, "cell_key": "3b9e89ab46895c84", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 7, "cell_key": "6066e0c6d042edab", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 2, "cell_key": "16b0f8f6dd985561", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 7, "cell_key": "4ebf0450d2c7f689", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 2, "cell_key": "2aadc1639802946a", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 7, "cell_key": "f59bb89cc889aa86", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 2, "cell_key": "3ec30c760c5b41cb", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 7, "cell_key": "a89ded7e7f514e78", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 2, "cell_key": "afa23753ba807bbe", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 7, "cell_key": "91fe47e9efa4f294", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 2, "cell_key": "77f99ae53209c854", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 7, "cell_key": "9c31e3057e8a5106", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "8641daffe0df27cd", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "92462d90d1eeb4c6", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "f76d63646adfc63e", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "9f5ef6d00df523ef", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "19dd4fcc577d9c78", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "25362e3ee2f155fc", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "0f988efa0f78f3ae", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "2c1f2055b9360185", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "f8c10984b248a25b", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "3fe2e1eadb9d41b4", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "fdf1882c85ad8cc0", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "4421ddfc1116fddd", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "69400aa0b282b457", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "41fc962b780f7403", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "fe2ee0589667c151", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "98857cd9cf0a6818", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "7573bb497604fd85", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "31ff32a929556d5e", "test_seed": 900007}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "c67f77a8f64c3aa6", "test_seed": 900002}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "88138d62be3a599c", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "a9ba3a0ff9ad5e7c", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "6230d469e63229d2", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "8ff381c4efbccddf", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "10d30c6b624a29ef", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "6ba89844b614d54b", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "c190086165fe6b32", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "07feff68d19638a4", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "fe680d59efc614c6", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "6620471445609cde", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "ffcadbb991b06b11", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "82d423d856cd3b6d", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "f6165b39d1eae7f4", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "b09d70f375719786", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "54f35f0449551f5c", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "6879028149c4f017", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "3fa60c654e8e3381", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 2, "cell_key": "49a0b9bbc9733656", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 7, "cell_key": "c13ef1bb935bf23e", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 2, "cell_key": "d4f6076f131673ec", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 7, "cell_key": "358b1552c792b24b", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 2, "cell_key": "716915e1562d76bf", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 7, "cell_key": "713121e139d92963", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 2, "cell_key": "a9f59b18333282e0", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 7, "cell_key": "5344f2a31c30bd4d", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 2, "cell_key": "7e1eb5a2c2817f19", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 7, "cell_key": "f681df0282bc2e59", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 2, "cell_key": "aa8f6c9146c471d7", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 7, "cell_key": "99d16d45a77d7f6b", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 2, "cell_key": "b3fbe00abf7cd1ef", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 7, "cell_key": "c6f95266a90d5e9a", "test_seed": 900007}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 2, "cell_key": "516b8e296770e1c0", "test_seed": 900002}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 7, "cell_key": "6b3433708342c20e", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "3fdba5318fc6aaa7", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "3628d9acd4b4b88b", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "5859216e4e476bb1", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "7f055912069aaeb1", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "45182480065bbb15", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "e5d89f53792051d0", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "c68b8584a1fafb82", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "27f088ac3c1edbbf", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "9f22e1624284ae37", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "0aa3c92380473471", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "bc23018ebf689c3e", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "b572362b17069349", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "f87b28c80d1afde5", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "8c94051956e229db", "test_seed": 900007}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "cddabb468dd89e0b", "test_seed": 900002}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "591da595928d9876", "test_seed": 900007}]''')
print(f'{len(CELLS)} cells in this shard')
for key, count in sorted(collections.Counter(
        (c['grid'], c['dgp'], c['method'])
        for c in CELLS).items()):
    print(f'  {key[0]:12s} {key[1]:8s} {key[2]:18s} {count}')

## 4. Bandwidth-selection pilot (preregistered)

This shard contains `causal_drf_retn` cells, so it first runs the selection pilot on seeds 100 and 101, outside every decisive range, and freezes the multipliers document. The rule picks the candidate with the best held-out energy score; oracle truth is never read.

In [ ]:
from pathlib import Path
import json, numpy as np
from wasserstein_causal_forests.g3.dgps import build_dgp
from wasserstein_causal_forests.g3.phase65_methods import (
    BANDWIDTH_CANDIDATES, SELECTION_SEEDS, select_bandwidth_multiplier,
)

keys = sorted({(c['dgp'], c['n_train']) for c in CELLS
               if c['method'] == 'causal_drf_retn'})
multipliers = {}
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)
for dgp_name, n_train in keys:
    dgp = build_dgp(dgp_name, 25)
    best, means = select_bandwidth_multiplier(
        dgp, n_train, seeds=SELECTION_SEEDS,
        candidates=BANDWIDTH_CANDIDATES, cache_directory=cache,
    )
    multipliers[f'{dgp_name}|{n_train}'] = best
    scores = {str(k): round(v, 5) for k, v in means.items()}
    print(f'{dgp_name} n={n_train}: multiplier {best}  scores {scores}',
          flush=True)

document = {
    'rule': 'held-out energy score, pilot seeds 100 and 101, '
            'candidates ' + repr(BANDWIDTH_CANDIDATES),
    'multipliers': multipliers,
}
path = Path('/content/wcf/results/manifests/'
            'phase65_bandwidth_selection.json')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(document, indent=2))
print('froze', path)

## 5. Run

In [ ]:
import time
from pathlib import Path
from wasserstein_causal_forests.g3.manifest import Cell
from wasserstein_causal_forests.g3.runner import run_shard

cells = [Cell(**{k: v for k, v in item.items()
                 if k not in ('cell_key', 'test_seed')})
         for item in CELLS]

out = Path('/content/wcf/results/phase65/colab_shards')
out.mkdir(parents=True, exist_ok=True)
log = Path(f'/content/wcf/results/manifests/phase65_execution_log_{SHARD_INDEX:03d}.jsonl')
log.parent.mkdir(parents=True, exist_ok=True)
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)

started = time.time()
summary = run_shard(
    cells,
    out / f'shard_{SHARD_INDEX:03d}.parquet',
    cache_directory=cache,
    log_path=log,
    manifest_contract_id='G3-PHASE65-v1',
)
print(json.dumps(summary, indent=2))
print(f'elapsed {(time.time() - started) / 60:.1f} min')

## Check

Every cell must appear exactly once, as a success or as a failure. Failures are kept and reported at merge time; a seed is never silently replaced.

In [ ]:
import collections
records = [json.loads(line) for line in
           open(log, encoding='utf-8') if line.strip()]
status = collections.Counter(r['status'] for r in records)
print('cells logged:', len(records), '| expected:', len(CELLS))
print('status:', dict(status))
assert len(records) == len(CELLS), 'shard did not finish every cell'
for record in records:
    if record['status'] != 'ok':
        print('  FAILED', record['dgp'], record['method'],
              record['seed'])
slowest = sorted(records, key=lambda r: -r['wall_seconds'])[:5]
print('slowest cells:', [(r['method'], round(r['wall_seconds'], 1))
                         for r in slowest])

## Download the results

In [ ]:
import shutil
bundle = '/content/p65_shard_14_forest65'
staging = Path('/content/bundle')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copy(out / f'shard_{SHARD_INDEX:03d}.parquet', staging)
if log.exists():
    shutil.copy(log, staging)
output_file = shutil.make_archive(bundle, 'zip', staging)
print('bundle:', output_file,
      f'({os.path.getsize(output_file) / 1e6:.2f} MB)')

try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)